# Этап 2 — исследование сбора базы знаний

Самостоятельный исследовательский ноутбук: пересчитывает результаты реально проведённых опытов из сохранённых данных. **Нет сетевых запросов, доступа к секретам и платных вызовов.** Встроены корпус, реестр источников, запросы/ответы моделей, эталон, версии кода. Результаты Python-проверок ниже исполнены локально; платные опыты ранее выполнены в Colab 22.10.2026. Для повторения проверки достаточно Run all.

Профиль → запросы исследуется отдельно в [первом Colab](https://colab.research.google.com/drive/1VrOW30seMq_-kOrXQ0pgJ9kdHI9FqX6S). [Основной Colab](https://colab.research.google.com/drive/1wWOfqx-i27wJDFxnG0SPkai_XFgwIqiN) хранит исходные исполнения. Этот ноутбук относится только к этапу 2; объединение этапов в приложение — последующая работа.

In [1]:
#@title Распаковать сохранённую базу и проверить манифест
from pathlib import Path
import base64,hashlib,io,json,zipfile,sys,lzma
ROOT=Path("/content/stage2_research") if Path("/content").exists() else Path.cwd()/"stage2_research"
ROOT.mkdir(parents=True,exist_ok=True)
DATA=lzma.decompress(base64.b64decode((
'...'
)))
assert hashlib.sha256(DATA).hexdigest()=='b7b3974cf2a30d35a6abcb6d7828a4e20aa1d616a8e3a6000ccbbe35d5863549'
with zipfile.ZipFile(io.BytesIO(DATA)) as archive:
    for name in archive.namelist():
        if not (ROOT/name).resolve().is_relative_to(ROOT.resolve()): raise ValueError("Unsafe archive path")
    archive.extractall(ROOT)
for item in json.loads((ROOT/"dataset_manifest.json").read_text(encoding="utf-8")):
    assert hashlib.sha256((ROOT/item["path"]).read_bytes()).hexdigest()==item["sha256"],item["path"]
sys.path.insert(0,str(ROOT))
print("Проверены все файлы манифеста:",len(json.loads((ROOT/"dataset_manifest.json").read_text(encoding="utf-8"))))


Проверены все файлы манифеста: 169


## Сбор и очистка Python

Проверяются байты/хэши HTML и очищенных текстов, смещения фрагментов. Неуспешные источники остаются в реестре. Повторные снимки одного URL не считаются разными компаниями. Публичный Telegram — ограниченное превью, отдельно от сайтов.

In [2]:
from research_audit import audit
RESULT=audit(ROOT)
print(json.dumps({k:RESULT[k] for k in ("source_rows","source_statuses","chunks_in_colab_corpus","same_inputs_across_experiments","integrity_errors","checks_passed")},ensure_ascii=False,indent=2))
assert RESULT["checks_passed"]


{
  "source_rows": 16,
  "source_statuses": {
    "ok": 15,
    "robots_disallowed_or_unavailable": 1
  },
  "chunks_in_colab_corpus": 21,
  "same_inputs_across_experiments": true,
  "integrity_errors": [],
  "checks_passed": true
}


## Извлечение LLM: сравнение сохранённых опытов

Один корпус и два контрольных фрагмента. Для v1/v2 меняется промпт на одной Luna; для сравнения моделей фиксирован evidence_v2. В v3 изменены промпт и валидатор одновременно — отдельный опыт доработки. Время ниже — весь опыт, а не индивидуальная задержка API. Тарифная оценка не равна фактическому списанию.

In [3]:
for c in RESULT["experiments"]:
    print(c["experiment"],c["model"],"цены",str(c["reference_result"]["matched_by_rules"])+"/8","принято",c["facts_accepted"],"карантин",len(c["quarantine"]),"ошибки",len(c["failures"]),"сек",c["elapsed_pipeline_seconds"],"оценка RUB",c["cost"]["tariff_estimate_rub"])


20261022T111947Z_adab53 openai/gpt-6-luna цены 8/8 принято 158 карантин 7 ошибки 0 сек 97.024 оценка RUB 2.40309
20261022T112126Z_2421a6 openai/gpt-6-luna цены 8/8 принято 76 карантин 7 ошибки 0 сек 74.33 оценка RUB 1.65024
20261022T112655Z_3a2e17 anthropic/claude-haiku-4-5 цены 3/8 принято 11 карантин 12 ошибки 1 сек 68.441 оценка RUB 13.18816
20261022T112655Z_fabfd2 google/gemini-3.5-flash-lite цены 0/8 принято 7 карантин 58 ошибки 0 сек 22.436 оценка RUB 7.929751
20261022T113037Z_be7ea6 openai/gpt-6-luna цены 8/8 принято 175 карантин 3 ошибки 0 сек 134.865 оценка RUB 2.90517


## Полные промпты, версии и ошибки

Каждая карточка содержит цель, модель, настройки, версию кода, хэши корпуса/промпта, даты UTC, время, usage, тарифную оценку и ошибки. Для ранних опытов исходник восстановлен из архива среды; хэш во время выполнения не журналировался. Полные пользовательские сообщения и схема сохранены в *_request.json.

In [4]:
for c in RESULT["experiments"]:
    print("\nОПЫТ",c["experiment"],"\nЦЕЛЬ",c["goal"],"\nКОД",c["code_version"],"\nНАСТРОЙКИ",json.dumps(c["settings"],ensure_ascii=False),"\nСИСТЕМНЫЙ ПРОМПТ\n",c["prompt"],"\nОШИБКИ",json.dumps(c["failures"],ensure_ascii=False))



ОПЫТ 20261022T111947Z_adab53 
ЦЕЛЬ Compare prompts 
КОД {'collector_sha256': 'efa273fb213d9dc5d6c49f10e29c8a0ab8789c325e19761da3befd704a0f1cfc', 'snapshot': 'review/colab_all_models_20261022.zip', 'path': 'code_versions/efa273fb213d9dc5d6c49f10e29c8a0ab8789c325e19761da3befd704a0f1cfc/collector.py', 'evidence': 'recovered_runtime_snapshot_no_runtime_hash'} 
НАСТРОЙКИ {"base_url": "https://api.proxyapi.ru/v1", "response_format": "json_schema", "parameters": {}, "chunk_ids": ["immigration_relocator_a317a051e73c04_c000", "migrun_374ee81f00c9b3_c000"]} 
СИСТЕМНЫЙ ПРОМПТ
 Извлеки из предоставленного фрагмента только явно опубликованные сведения о компании. Верни JSON по схеме: facts. Каждая запись должна содержать field, value, evidence, kind, country, offer_name, amount, currency, price_basis. Цитата evidence должна быть точной непустой подстрокой текста. Не используй внешние знания. Отделяй обещания компании (company_claim) от проверяемых элементов предложения (published_offer). Не выводи

## База и ручной контроль

Выбранный v3 дал 175 записей после автоматической проверки и 3 в карантине. Это **черновая база**, полного смыслового утверждения нет. Восемь контрольных цен подготовлены вручную до LLM; совпадение по правилам не измеряет точность всех 175 фактов. Контроль налоговой консультации сохраняет обе суммы 170/350 EUR, не выбирая актуальную автоматически.

In [5]:
CARD=json.loads((ROOT/"knowledge_base/dataset_card.json").read_text(encoding="utf-8"))
print(json.dumps(CARD,ensure_ascii=False,indent=2))
print("\nКонтрольные примеры:")
for row in json.loads((ROOT/"review/reference.json").read_text(encoding="utf-8")):
    print(row["reference_id"],row["company_id"],row["amount"],row["manual_note"])


{
  "status": "pilot_draft",
  "own_company": "Immigration Relocator",
  "competitors_in_detailed_pilot": [
    "Migrun"
  ],
  "confirmed_competitor_list": "competitor_candidates.json",
  "website_snapshots": [
    "20261021T191729Z_4005ca",
    "20261022T111451Z_0f8116"
  ],
  "facts_selected_experiment": "20261022T113037Z_be7ea6",
  "facts_count": 175,
  "quarantine_count": 3,
  "social_posts_count": 25,
  "manual_price_references": 8,
  "facts_semantic_review": "not_complete",
  "source_claims_independently_verified": false,
  "scope_limit": "Two companies; website source rows include repeated snapshots. Eight confirmed competitors have not all been collected.",
  "actual_api_billing_rub": null
}

Контрольные примеры:
IR01 immigration_relocator 500 Опубликованная цена услуги; проверена по сохранённому тексту, не коммерческое предложение.
IR02 immigration_relocator 150 Не смешивать с налоговой консультацией.
IR03 immigration_relocator 170 Первый блок. На той же странице есть 350 EUR

## Вывод и остаток работы

Luna — предварительный выбор для извлечения в этом пилоте: 8/8 цен при меньшей тарифной оценке. Gemini нарушила разделение суммы и поля услуги; Claude получила length при 4096 токенах на одной странице. Для честного следующего опыта нужны отдельные адаптации, а не вывод о качестве вендоров вообще. v2 дешевле v1, но теряет часть отдельных полей; v3 исправил наблюдавшийся дефект цитаты о типах виз и вернул отдельные поля, стоит дороже.

Остаются: смысловая проверка фактов, расширение эталона за пределы цен, автоматический поиск выдачи/кандидатов, полный сбор восьми конкурентов, проверка соцсетей и карт, рекомендации. Профиль/кандидаты ранее подтверждены пользователем; повторное подтверждение тех же вводных не требуется.

Промежуточный контракт объединения этапов: knowledge_base/facts_draft.json + sources.json + dataset_card.json; только утверждённые человеком факты допускаются в будущие рекомендации. Схема интеграции ещё не реализована как приложение.